# 1. Business Understanding

## 1.1 Problem Statement
The goal of this project is to develop a machine learning model that predicts the **primary contributory cause of traffic crashes** using data related to road conditions, vehicles, and individuals involved in Chicago road crashes.

## 1.2 Objectives
- Build a classification model to predict crash causes
- Identify key factors contributing to traffic accidents
- Provide insights that can help reduce accidents

## 1.3 Stakeholders
- Chicago City planners → improve road safety
- Chicago Traffic safety authorities → reduce accident rates
- Chicago Policy makers → design effective interventions

## 1.4 Analytical Approach
This problem is framed as a **multi-class classification task**, where the target variable is:

PRIM_CONTRIBUTORY_CAUSE

## 1.5 Success Criteria
The success of the model will be evaluated using:
- Macro F1-score (to handle class imbalance)
- Confusion matrix (to understand misclassifications)
- Interpretability (feature importance analysis)

## 1.6 Key Challenges
- Imbalanced target classes
- High number of categorical variables
- Data spread across multiple tables (crashes, vehicles, people)
- Risk of data leakage

# 2. Data Understanding

## 2.1 Dataset Description
This project uses three datasets provided by the City of Chicago:

- **Crashes Dataset**: Contains information about each traffic crash (e.g., time, location, weather conditions, road conditions).
- **Vehicles Dataset**: Contains details about the vehicles involved in each crash.
- **People Dataset**: Contains information about drivers and passengers involved in each crash.

These datasets are related through a common key: `CRASH_RECORD_ID`.

## 2.2 Objective of This Section
- Load the datasets
- Understand their structure
- Identify key columns
- Perform initial exploration

In [8]:
#import pandas library 
import pandas as pd



# Load datasets
crashes = pd.read_csv("Data/Raw/Traffic_Crashes.csv", index_col=0, low_memory=False)
vehicles = pd.read_csv("Data/Raw/vehicle_data.csv", index_col=0, low_memory=False)
people = pd.read_csv("Data/Raw/passenger_driver.csv", index_col=0, low_memory=False)

# Display shapes
print("Crashes shape:", crashes.shape)
print("Vehicles shape:", vehicles.shape)
print("People shape:", people.shape)

Crashes shape: (251295, 47)
Vehicles shape: (511366, 70)
People shape: (550849, 28)


## 2.3 Data Structure Inspection

Before performing exploratory analysis, we examine the structure of each dataset, including:
- Column names and meanings
- Data types
- Missing values
- Summary statistics

This helps identify data quality issues and informs preprocessing decisions.

In [9]:
print("Crashes Columns:")
print(crashes.columns.tolist())

print("\nVehicles Columns:")
print(vehicles.columns.tolist())

print("\nPeople Columns:")
print(people.columns.tolist())

Crashes Columns:
['CRASH_DATE_EST_I', 'CRASH_DATE', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'LANE_CNT', 'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE', 'INTERSECTION_RELATED_I', 'NOT_RIGHT_OF_WAY_I', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED', 'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO', 'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'PHOTOS_TAKEN_I', 'STATEMENTS_TAKEN_I', 'DOORING_I', 'WORK_ZONE_I', 'WORK_ZONE_TYPE', 'WORKERS_PRESENT_I', 'NUM_UNITS', 'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING', 'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION', 'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'LATITUDE', 'LONGITUDE', 'LOCATION']

Vehicles Columns:
['CRASH_RECORD_ID', 'CRASH_DATE', 'UNIT_NO', 'UNIT_TYPE', 'NUM

In [10]:
# crashes data types and missing values
crashes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 251295 entries, 46ea518f2e11a7cb35a3af47503b0e9712237347e0a6d61a3d661fd1f75239b12bba6d5d8af0892a65fd7fb61ca52f51b2031b8562af4e05520089e6cf6710e9 to ce46f735c5c4d216ef355f25b4159dbc97637fbb3431b0d37ef542e1631f8507d053e7c0a0c1788ef64b20ffb43cce956820137ba3acec72e9bce86b19a05a65
Data columns (total 47 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH_DATE_EST_I               16190 non-null   object 
 1   CRASH_DATE                     251295 non-null  object 
 2   POSTED_SPEED_LIMIT             251295 non-null  int64  
 3   TRAFFIC_CONTROL_DEVICE         251295 non-null  object 
 4   DEVICE_CONDITION               251295 non-null  object 
 5   WEATHER_CONDITION              251295 non-null  object 
 6   LIGHTING_CONDITION             251295 non-null  object 
 7   FIRST_CRASH_TYPE               251295 non-null  object 
 8   TRAFFICWAY_TYPE               

In [11]:
# vehicles data types and missing values
vehicles.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 511366 entries, 2284891 to 1733631
Data columns (total 70 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   CRASH_RECORD_ID           511366 non-null  object 
 1   CRASH_DATE                511366 non-null  object 
 2   UNIT_NO                   511366 non-null  int64  
 3   UNIT_TYPE                 511020 non-null  object 
 4   NUM_PASSENGERS            74864 non-null   float64
 5   VEHICLE_ID                497726 non-null  float64
 6   CMRC_VEH_I                9295 non-null    object 
 7   MAKE                      497726 non-null  object 
 8   MODEL                     497726 non-null  object 
 9   LIC_PLATE_STATE           453010 non-null  object 
 10  VEHICLE_YEAR              427944 non-null  float64
 11  VEHICLE_DEFECT            497726 non-null  object 
 12  VEHICLE_TYPE              497726 non-null  object 
 13  VEHICLE_USE               497726 non-

In [12]:
# people data types and missing values
people.info()

<class 'pandas.core.frame.DataFrame'>
Index: 550849 entries, O2284891 to O1733630
Data columns (total 28 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   PERSON_TYPE            550849 non-null  object 
 1   CRASH_RECORD_ID        550849 non-null  object 
 2   VEHICLE_ID             537572 non-null  float64
 3   CRASH_DATE             550849 non-null  object 
 4   SEAT_NO                109115 non-null  float64
 5   CITY                   402011 non-null  object 
 6   STATE                  409448 non-null  object 
 7   ZIPCODE                380516 non-null  object 
 8   SEX                    540181 non-null  object 
 9   AGE                    394788 non-null  float64
 10  DRIVERS_LICENSE_STATE  325326 non-null  object 
 11  DRIVERS_LICENSE_CLASS  255549 non-null  object 
 12  SAFETY_EQUIPMENT       549376 non-null  object 
 13  AIRBAG_DEPLOYED        538046 non-null  object 
 14  EJECTION               543045 no

### Key Observations from Data Structure Inspection

- The crashes dataset contains 251,295 records with 47 features, representing crash-level information such as weather, lighting, road conditions, and crash outcomes.
- The vehicles dataset contains 511,366 records with 70 features, indicating that multiple vehicles can be involved in a single crash.
- The people dataset contains 550,849 records with 28 features, showing that multiple individuals are associated with each crash.

- The datasets are linked using the `CRASH_RECORD_ID`, confirming a relational structure.

- The crashes dataset contains a mix of numerical and categorical variables, with several columns having significant missing values.

- The vehicles dataset has many columns related to commercial vehicles and hazardous materials, most of which have very high missing values and may not be useful for this analysis.

- The people dataset includes detailed driver and passenger information, but also contains several columns with substantial missing data.

These observations will guide feature selection, aggregation, and data cleaning in subsequent steps.

## 2.4 Exploratory Data Analysis (EDA)

In this section, we explore the datasets to understand:
- Distribution of the target variable
- Key patterns in crash conditions
- Potential relationships between features and crash causes
- Data quality issues such as missing values and inconsistencies